# Privacy-Preserving Record Linkage using Bloom filters and Similarity threshold-based classification

In [47]:
# Imports
#
import time
import math
from BF import BF #import the BF module
from PPRL import Link #import the PPRL module

In [48]:
start_time = time.time()

In [49]:
#Create an instance of class Link with the following parameter values
#Modify the values below to fine-tune the hyper-parameters
#
BF_length = 1000
BF_num_hash = 10
BF_q_gram = 2
min_sim_val = 0.8
link_attrs = [1,2,3,4]
block_attrs = [2,4]
ent_index = 0
epsilon = 7

link = Link(BF_length,BF_num_hash,BF_q_gram,min_sim_val,link_attrs,block_attrs,ent_index,epsilon)

In [50]:
#Read the first dataset - provide the link for the first dataset
db1 = link.read_database('Datasets/Alice_numrec_100_non_corr.csv')
#print(db1)

Load data file: Datasets/Alice_numrec_100_non_corr.csv
Read 100 records


In [51]:
#Read the second dataset - provide the link for the second dataset
db2 = link.read_database('Datasets/Bob_numrec_100_non_corr.csv')
#print(db2)

Load data file: Datasets/Bob_numrec_100_non_corr.csv
Read 100 records


In [52]:
#Apply blocking on the two datasets
blk_ind1 = link.build_BI(db1)
blk_ind2 = link.build_BI(db2)

Build Block Index for attributes: [2, 4]
Generate 97 blocks
Build Block Index for attributes: [2, 4]
Generate 89 blocks


In [53]:
#Encode records into Bloom filters
bf_dict1, all_val_set1 = link.data_encode(db1)
bf_dict2, all_val_set2 = link.data_encode(db2)

#Calculate false positive rate of bloom filter encoding
all_val_set = all_val_set1 + all_val_set2
total_all_val_set = set(all_val_set)
num_total_all_val_set = len(total_all_val_set)

fpr = (1 - math.e**((-1*BF_num_hash*num_total_all_val_set)/BF_length))**BF_num_hash
print(fpr)

0.8746852331237396


In [54]:
#Add bit-level differential privacy noise to Bloom filters
pbf_dict1 = link.add_DP_noise(bf_dict1)
pbf_dict2 = link.add_DP_noise(bf_dict2)

In [55]:
#Match and link Bloom filters from the two datasets
matches = link.match(blk_ind1,blk_ind2,pbf_dict1,pbf_dict2) 

number of common blocks: 56
Number of matching pairs: 50


In [56]:
#Evaluate runtime
end_time = time.time() - start_time
print('Total time in seconds:', end_time)

Total time in seconds: 0.18317532539367676


In [57]:
#Evaluate linkage quality
print('Linkage quality of PPRL')
prec, rec, f1 = link.evaluate(matches,db1,db2)
print('Probable Privacy guarantees:', 'false positive rate of Bloom filters (larger better) - ', fpr)
print('Provable Privacy guarantees:', 'Privacy budget (smaller better) - ', epsilon)

Linkage quality of PPRL
Precision:  1.0
Recall:  1.0
F1 score:  1.0
Probable Privacy guarantees: false positive rate of Bloom filters (larger better) -  0.8746852331237396
Provable Privacy guarantees: Privacy budget (smaller better) -  7


In [58]:
#Baseline: Macth and link from two datasets using non-privacy-preserving record linkage
matches_npp = link.match_npp(blk_ind1,blk_ind2,db1,db2)

number of common blocks: 56
Number of matching pairs: 50


In [59]:
#Baseline2: Match and link Bloom filters from two datasets without DP guarantees
matches_nodp = link.match(blk_ind1,blk_ind2,bf_dict1,bf_dict2)

number of common blocks: 56
Number of matching pairs: 50


In [60]:
#Evaluate linkage quality of non-privacy-preserving record linkage baseline method
print('Linkage quality of non-PPRL')
prec_b1, rec_b1, f1_b1 = link.evaluate(matches_npp,db1,db2)
print('Privacy guarantees:', 'None')

Linkage quality of non-PPRL
Precision:  1.0
Recall:  1.0
F1 score:  1.0
Privacy guarantees: None


In [61]:
#Evaluate linkage quality of privacy-preserving record linkage without Differential privacy guarantees
print('Linkage quality of PPRL without DP')
prec_b2, rec_b2, f1_b2 = link.evaluate(matches_nodp,db1,db2)
print('Probable Privacy guarantees:', 'false positive rate of Bloom filters (larger better) - ', fpr)
print('Provable Privacy guarantees:', 'None')

Linkage quality of PPRL without DP
Precision:  1.0
Recall:  1.0
F1 score:  1.0
Probable Privacy guarantees: false positive rate of Bloom filters (larger better) -  0.8746852331237396
Provable Privacy guarantees: None


In [62]:
# KNN linkage
k = 3
matches_knn = link.match_knn(
    blk_ind1,
    blk_ind2,
    pbf_dict1,
    pbf_dict2,
    db1,
    db2,
    k=k,
 )

print('Number of KNN matching pairs:', len(matches_knn))
print('KNN linkage quality')
prec_knn, rec_knn, f1_knn = link.evaluate(matches_knn, db1, db2)

number of common blocks: 56
Number of matching pairs: 50
Number of KNN matching pairs: 50
KNN linkage quality
Precision:  1.0
Recall:  1.0
F1 score:  1.0


In [63]:

# Random Forest Classifier for Privacy-Preserving Record Linkage


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np


pair_features = []
pair_labels = []

for blk in blk_ind1:
    if blk in blk_ind2:
        for rec1 in blk_ind1[blk]:
            for rec2 in blk_ind2[blk]:
                bf1 = pbf_dict1[rec1]
                bf2 = pbf_dict2[rec2]
                sim = link.bf.calc_bf_sim(bf1, bf2)
                pair_features.append([sim])
                # Label: 1 if true match, 0 if not
                if db1[rec1][0] == db2[rec2][0]:
                    pair_labels.append(1)
                else:
                    pair_labels.append(0)

X = np.array(pair_features)
y = np.array(pair_labels)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


y_pred = rf_model.predict(X_test)

rf_precision = precision_score(y_test, y_pred, zero_division=0)
rf_recall = recall_score(y_test, y_pred, zero_division=0)
rf_f1 = f1_score(y_test, y_pred, zero_division=0)

print('Linkage quality of Random Forest PPRL')
print('Precision: ', rf_precision)
print('Recall: ', rf_recall)
print('F1 score: ', rf_f1)

Linkage quality of Random Forest PPRL
Precision:  1.0
Recall:  1.0
F1 score:  1.0


In [64]:
!pip install scikit-learn


In [65]:
matches_svm = link.match_svm(
    blk_ind1,
    blk_ind2,
    pbf_dict1,
    pbf_dict2,
    db1,
    db2
)

print("SVM linkage quality")
link.evaluate(matches_svm, db1, db2)

Number of matching pairs (SVM): 50
SVM linkage quality
Precision:  1.0
Recall:  1.0
F1 score:  1.0


(1.0, 1.0, 1.0)

In [66]:
matches_svm = link.match_svm(
    blk_ind1,
    blk_ind2,
    pbf_dict1,
    pbf_dict2,
    db1,
    db2
)

print("SVM linkage quality")
link.evaluate(matches_svm, db1, db2)

Number of matching pairs (SVM): 50
SVM linkage quality
Precision:  1.0
Recall:  1.0
F1 score:  1.0


(1.0, 1.0, 1.0)

In [67]:
matches_lr = link.match_lr(blk_ind1, blk_ind2, pbf_dict1, pbf_dict2, db1, db2)

print("LR linkage quality")
link.evaluate(matches_lr, db1, db2)

number of common blocks: 56
Number of matching pairs: 56
LR linkage quality
Precision:  0.8928571428571429
Recall:  1.0
F1 score:  0.9433962264150945


(0.8928571428571429, 1.0, 0.9433962264150945)

In [68]:
!pip install scikit-learn matplotlib
